# Copernicus Marine Delivery – Python demo

User `mds_test_2` (Mercator, no `pushing_entity_id` needed). Same flow as `cli.md`.

**Before running:** `demo/clean_native.sh` then `demo/seed_delete_targets.sh` (delete targets must exist).

In [1]:
import os
from pathlib import Path

# Load credentials from the repo .env ("export KEY=value" lines) if not already set.
# Skip PYTHON_LOG_DEBUG: the demo runs with INFO logs.
for line in Path("../.env").read_text().splitlines():
    if line.startswith("export ") and "=" in line:
        key, value = line[len("export "):].split("=", 1)
        if key != "PYTHON_LOG_DEBUG":
            os.environ.setdefault(key, value.strip('"'))

# yaml file paths are relative to the CWD
os.chdir("delivery_multiple_operations")
os.environ.pop("PYTHON_LOG_DEBUG", None)
print(os.environ["COPERNICUSMARINE_SERVICE_USERNAME"], Path.cwd())

mds_test_2 /Users/hmuro/dev/copernicusmarine-delivery/demo/delivery_multiple_operations


In [2]:
PRODUCT_ID = "GLOBAL_ANALYSISFORECAST_PHY_001_024"
DATASET_ID = "cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i_202406"

## 1. Build the delivery

A delivery is a list of operations that run in order. Delete 4 files, then upload 16 (4 of them re-use a deleted name).

In [3]:
from copernicusmarine_delivery import Delete, Delivery, Upload, delivery_status, list_deliveries

delivery = Delivery()

# 1st operation: delete files already published in the dataset (S3 key suffixes)
delivery.add(
    Delete(
        files=[
            "2026/05/glo12_rg_6h-i_20260528-18h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260529-00h_3D-uovo_nwct_R20260603.nc",
            "2026/06/glo12_rg_6h-i_20260602-06h_3D-uovo_nwct_R20260603.nc",
            "2026/06/glo12_rg_6h-i_20260602-12h_3D-uovo_nwct_R20260603.nc",
        ]
    )
)

# 2nd operation: upload local files (all .nc files below 2026/)
delivery.add(Upload(files=[str(f) for f in sorted(Path("2026").rglob("*.nc"))]))

for op in delivery.operations:
    print(type(op).__name__, len(op.files), "files")

Delete 4 files
Upload 16 files


## 2. Dry run

Validates product, dataset and files. Nothing is uploaded.

In [4]:
response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID, dry_run=True)
print("Fatal error:", response.fatal_error)

2026-10-29 10:16:10 - copernicusmarine-delivery - INFO - [DRY RUN]: Starting to upload 16 files ...
2026-10-29 10:16:11 - copernicusmarine-delivery - INFO - [DRY RUN]: Finished uploading 16 files for a total size of 118.00 MB.


Fatal error: None


## 3. Submit

In [5]:
response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)

if response.fatal_error:
    print(f"Delivery failed: {response.fatal_error}")
else:
    print(f"Delivery ID: {response.delivery_id}")
    for r in response.operations_responses:
        print(type(r).__name__)
        print("  files to delete:", getattr(r, "files_to_delete", None))
        print("  files uploaded:", getattr(r, "files_uploaded", None))
        print("  files failed:", getattr(r, "files_failed", None))

2026-10-29 10:16:23 - copernicusmarine-delivery - INFO - Starting to upload 16 files ...
2026-10-29 10:16:24 - copernicusmarine-delivery - INFO - Uploaded [1/16] file glo12_rg_6h-i_20260527-18h_3D-uovo_nwct_R20260603.nc
2026-10-29 10:16:24 - copernicusmarine-delivery - INFO - Uploaded [2/16] file glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc
2026-10-29 10:16:24 - copernicusmarine-delivery - INFO - Uploaded [3/16] file glo12_rg_6h-i_20260528-12h_3D-uovo_nwct_R20260603.nc
2026-10-29 10:16:24 - copernicusmarine-delivery - INFO - Uploaded [4/16] file glo12_rg_6h-i_20260528-06h_3D-uovo_nwct_R20260603.nc
2026-10-29 10:16:25 - copernicusmarine-delivery - INFO - Uploaded [5/16] file glo12_rg_6h-i_20260528-00h_3D-uovo_nwct_R20260603.nc
2026-10-29 10:16:25 - copernicusmarine-delivery - INFO - Uploaded [6/16] file glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc
2026-10-29 10:16:25 - copernicusmarine-delivery - INFO - Uploaded [7/16] file glo12_rg_6h-i_20260529-06h_3D-uovo_nwct_R202606

Delivery ID: 20261029T101623-GLOBAL_ANALYSISFORECAST_PHY_001_024-8840
ResponseDelete
  files to delete: [DeleteFile(key_suffix='2026/05/glo12_rg_6h-i_20260528-18h_3D-uovo_nwct_R20260603.nc', error_message=None), DeleteFile(key_suffix='2026/05/glo12_rg_6h-i_20260529-00h_3D-uovo_nwct_R20260603.nc', error_message=None), DeleteFile(key_suffix='2026/06/glo12_rg_6h-i_20260602-06h_3D-uovo_nwct_R20260603.nc', error_message=None), DeleteFile(key_suffix='2026/06/glo12_rg_6h-i_20260602-12h_3D-uovo_nwct_R20260603.nc', error_message=None)]
  files uploaded: None
  files failed: None
ResponseUpload
  files to delete: None
  files uploaded: [(PosixPath('2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc'), 'GLOBAL_ANALYSISFORECAST_PHY_001_024/cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i_202406/2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc'), (PosixPath('2026/05/glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc'), 'GLOBAL_ANALYSISFORECAST_PHY_001_024/cmems_mod_glo_phy-cur_anf

## 4. Status

Reuses the `delivery_id` from the submit above.

In [6]:
import time

def wait_for_status(delivery_id: str):
    """Poll until the delivery is completed or failed, then show errors."""
    for _ in range(10):
        status = delivery_status(delivery_id)
        if status.status.value in ("completed", "failed"):
            break
        time.sleep(10)
    print("Delivery error:", status.error_message)
    for op in status.operations:
        for key_suffix, error in op.get_errored_file_names():
            print(f"  {op.operation.value} {key_suffix}: {error}")
    return status

DELIVERY_ID = response.delivery_id
status = wait_for_status(DELIVERY_ID)

validated
completed
Delivery error: None


## 5. List deliveries

In [ ]:
for d in list_deliveries(timeframe="d"):
    print(d.delivery_id, d.status.value)

## Error cases

### A. Toolbox detects the error

Two of the files do not exist locally. The toolbox validates all files **before** uploading anything, so nothing is uploaded and no delivery is created.

In [ ]:
from copernicusmarine_delivery import InvalidFilesError

delivery_a = Delivery()
delivery_a.add(
    Upload(
        files=[
            "2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-18h_3D-uovo_nwct_R20260603.nc",
            "2026/05/this_file_does_not_exist_locally_1.nc",
            "2026/06/this_file_does_not_exist_locally_2.nc",
        ]
    )
)

try:
    delivery_a.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
except InvalidFilesError as error:
    print("Toolbox error:", error)

### B. OPDV detects the error

The toolbox cannot know what is in the dataset, so it accepts a delete of a file that does not exist. OPDV rejects it afterwards.

In [ ]:
delivery_b = Delivery()
delivery_b.add(Delete(files=["2026/05/glo12_rg_6h-i_20200101-00h_3D-uovo_nwct_DOES_NOT_EXIST.nc"]))
delivery_b.add(Upload(files=["2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc"]))

response_b = delivery_b.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
print(f"Delivery ID: {response_b.delivery_id}")
status_b = wait_for_status(response_b.delivery_id)

### C. Delivery OK, error in publication

Toolbox and OPDV accept the delivery. The API is hardcoded to fail the publication of the file named `*_PUBERROR.nc`.

In [ ]:
os.chdir("../delivery_publication_error")

delivery_c = Delivery()
delivery_c.add(
    Upload(
        files=[
            "2026/06/glo12_rg_6h-i_20260603-00h_3D-uovo_nwct_R20260604.nc",
            "2026/06/glo12_rg_6h-i_20260603-06h_3D-uovo_nwct_R20260604.nc",
            "2026/06/glo12_rg_6h-i_20260603-12h_3D-uovo_nwct_PUBERROR.nc",
        ]
    )
)

response_c = delivery_c.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
print(f"Delivery ID: {response_c.delivery_id}")
status_c = wait_for_status(response_c.delivery_id)